# Problem 3 — Compact publication results

1. **Figure 1:** mean PGM error for all five input states, using Exhaustive only.
2. **Figure 2:** grouped bars of mean error gaps for Greedy, VND, Near-Uniform, and Random relative to Exhaustive, for each input state.

Each figure contains 1D and 2D panels. Figures combine all selected instances. Numerical tables report overall and field-family summaries. Full mode includes every instance in the ordinary-random dataset; test mode selects a balanced subset.

Greedy allocates sensors to the least-separated field pair. VND independently starts near uniformly and maximizes the capped phase-separation score using transfers of one, two, and three sensors until convergence (tolerance $10^{-12}$). Neither heuristic evaluates PGM during allocation. Each returns one allocation per instance and budget, shared across all five states. Final comparisons use the unchanged global PGM error. Exhaustive is the Compact reference, optimal over allocations for the specified state and PGM measurement.

In [ ]:
from pathlib import Path
import sys, importlib, json
from collections import Counter
from IPython.display import display, Markdown
ROOT = Path.cwd().resolve()
if (ROOT/'problem-3-global-pgm').is_dir(): ROOT = ROOT/'problem-3-global-pgm'
assert (ROOT/'src/pgm.py').exists()
loaded = sys.modules.get('src')
if loaded and Path(loaded.__file__).resolve().parent != ROOT/'src':
    raise RuntimeError('Restart kernel: another problem uses the src package')
for name in list(sys.modules):
    if name=='src' or name.startswith('src.') or name in ('reports','publication'): del sys.modules[name]
sys.path.insert(0,str(ROOT))
importlib.invalidate_caches()
from src.analysis import METHODS, run_analysis, select_cases
from src.reporting.publication import prepare, figures
STUDY = 'compact'
DATASET_ROOT = ROOT/'inputs'/STUDY
RUN_MODE = 'full'       # 'full' uses all cases; 'test' uses a balanced sample.
RUN_ANALYSIS = True
RESUME = False
MAX_ALLOCATIONS = 20_000  # Compact maximum is C(20,4)=4845; unused by Extended.
TIME_LIMIT_SECONDS = 120.0
BOOTSTRAP_SEED = 20261014
BOOTSTRAP_RESAMPLES = 2000
SHOW_ZOOM = True
INCLUDE_REFERENCE_BASELINES = True


## Execution

Results are written to `results/compact/full/` or `results/extended/full/` (and separate `test/` folders). A fresh run overwrites the previous results for that study and mode. With `RESUME=True`, matching saved search records are reused and only missing runs are evaluated. Setting `RUN_ANALYSIS=False` loads matching saved results for reporting only. Plotting changes do not change the search cache signature; production source, dataset, and execution settings must still match.

Reference allocations are enabled by default. Near-Uniform and Random (one seeded allocation per instance and budget, assigning each sensor independently to a uniformly chosen location) appear in the second figure and summary tables. They are reference baselines, not optimization algorithms.


In [ ]:
import matplotlib.pyplot as plt

records, OUTPUT = run_analysis(
    STUDY, RUN_MODE, run=RUN_ANALYSIS, resume=RESUME,
    max_allocations=MAX_ALLOCATIONS, time_limit=TIME_LIMIT_SECONDS,
    dataset_root=DATASET_ROOT, include_baselines=INCLUDE_REFERENCE_BASELINES,
)
PUBLICATION_OUTPUT = OUTPUT / 'publication_greedy_vnd'
print('Preparing confidence intervals and summary tables...', flush=True)
report = prepare(records, STUDY, PUBLICATION_OUTPUT,
                 seed=BOOTSTRAP_SEED, resamples=BOOTSTRAP_RESAMPLES)
print('Publication output:', PUBLICATION_OUTPUT)


## Estimation and confidence intervals

All intervals are **pointwise 95% percentile bootstrap confidence intervals** from 2,000 seeded resamples of field instances, stratified by the predefined benchmark design cells. Repeated budgets, states, and method comparisons from a field instance form one sampling unit. Intervals are conditional on the benchmark selection procedure, not simultaneous confidence bands or evidence of statistical significance.

Curve points group target fractions 25%, 50%, 75%, and 100%. Their x coordinates are mean actual ratios $N/N_{\mathrm{LB}}^{\mathrm{joint}}$, which need not equal the target fractions when caps or rounding apply. Duplicate integer budgets are evaluated once. Figures 2 and 4 average paired error differences over unique budgets within each field instance, separately for each input state, before averaging instances. Each field instance has equal weight.

State figures require all five states to complete for the specified reference method. Gap figures require all compared methods and states to complete at each field/budget. Missing or timed-out runs are excluded explicitly in the coverage table. Full mode is intended for publication; test mode may have singleton sampling strata and degenerate confidence intervals.

Figures use Times New Roman at 12 pt, consistent colors and markers, vector PDF with embedded fonts, and 600-dpi PNG export. The 10.8-inch figure width follows the two-panel layout used in Problem 2.

In [ ]:
publication_figures = figures(report, STUDY, zoom=SHOW_ZOOM)
assert len(publication_figures) == 2
for figure in publication_figures:
    display(figure)
    plt.close(figure)

display(Markdown('**Error bars:** 95% percentile bootstrap confidence intervals for the mean paired error difference from Exhaustive (2,000 resamples; field instances stratified by K, M, and field family). Budgets are averaged within each instance.'))


## Numerical summaries

Tables summarize the complete benchmark and each field family separately. An optimality match means error within $10^{-7}$ of Exhaustive, not an identical allocation vector. VND wins, ties, or loses according to its paired error difference from Greedy with the same tolerance; percentages average unique budgets within each instance first. These comparisons remain available when Exhaustive is incomplete.

Runtime summaries separate allocation time per instance/budget from final PGM evaluation time per state. Allocation time is counted once across the five states; combined per-state time assigns one fifth of that cost to each state. Exhaustive reports its full enumeration/PGM search time separately because objective evaluation is intrinsic to its search. Runtime means describe successful runs; coverage and status CSVs retain excluded outcomes. Detailed figure coordinates, intervals, and table values are exported as CSV files.

In [ ]:
def show_table(rows):
    if not rows: print('No complete results'); return
    keys=list(rows[0]);lines=['| '+' | '.join(keys)+' |','| '+' | '.join(['---']*len(keys))+' |']
    for row in rows:
        lines.append('| '+' | '.join(f'{row[k]:.6g}' if isinstance(row[k],float) else str(row[k]) for k in keys)+' |')
    display(Markdown('\n'.join(lines)))

In [ ]:
for overall, heading in [(True, 'Overall summary'), (False, 'Field-family summary')]:
    display(Markdown('## '+heading))
    for key, title in [('quality', 'Error and gap summary'),
                       ('runtime', 'Runtime (seconds)'),
                       ('outcomes', 'VND versus Greedy')]:
        rows = [row for row in report[key] if (row['family'] == 'Overall') == overall]
        if overall:
            rows = [{k:v for k,v in row.items() if k != 'family'} for row in rows]
        display(Markdown('### '+title))
        show_table(rows)
